# Prediction of Product Sales

- Author: Salam Odeh

## Modeling



This notebook adds modeling to the sales prediction project. The goal is to help the retailer understand the properties of products and outlets that play crucial roles in predicting sales, by building and comparing machine learning models.



This notebook repeats the data loading, cleaning, and preprocessing steps from Part 5 so it can run standalone, then proceeds to:

1. Build and evaluate a **Linear Regression** model.

2. Build and evaluate a default **Random Forest** model.

3. Use `GridSearchCV` to tune the Random Forest model.

4. Evaluate which model performs best and explain the results to a non-technical stakeholder.

In [1]:
# Imports

import pandas as pd

import numpy as np



from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.impute import SimpleImputer

from sklearn.pipeline import make_pipeline

from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LinearRegression

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (mean_absolute_error, mean_squared_error,

                              root_mean_squared_error, r2_score)



from sklearn import set_config

set_config(transform_output='pandas')



pd.set_option('display.max_columns', 100)

## Custom Evaluation Function



We'll use a custom function to consistently report metrics for every model we build: **R²**, **MAE**, **MSE**, and **RMSE**.

In [2]:
def regression_metrics(y_true, y_pred, label='',

                        verbose=True, output_dict=False):

    """Calculate MAE, MSE, RMSE, and R^2 and optionally print/return them."""

    mae = mean_absolute_error(y_true, y_pred)

    mse = mean_squared_error(y_true, y_pred)

    rmse = root_mean_squared_error(y_true, y_pred)

    r_squared = r2_score(y_true, y_pred)



    if verbose == True:

        header = "-" * 60

        print(header, f"Regression Metrics: {label}", header, sep='\n')

        print(f"- MAE  = {mae:,.3f}")

        print(f"- MSE  = {mse:,.3f}")

        print(f"- RMSE = {rmse:,.3f}")

        print(f"- R\u00b2   = {r_squared:,.3f}")



    if output_dict == True:

        metrics = {'Label': label, 'MAE': mae, 'MSE': mse,

                   'RMSE': rmse, 'R\u00b2': r_squared}

        return metrics





def evaluate_regression(reg, X_train, y_train, X_test, y_test,

                         verbose=True, output_frame=False):

    """Print (and optionally return as a DataFrame) train/test metrics."""

    y_train_pred = reg.predict(X_train)

    results_train = regression_metrics(

        y_train, y_train_pred, verbose=verbose,

        output_dict=output_frame, label='Training Data')



    print()



    y_test_pred = reg.predict(X_test)

    results_test = regression_metrics(

        y_test, y_test_pred, verbose=verbose,

        output_dict=output_frame, label='Test Data')



    if output_frame:

        results_df = pd.DataFrame([results_train, results_test])

        results_df = results_df.set_index('Label')

        results_df.index.name = None

        return results_df.round(3)

## Load, Clean, and Split the Data



(Repeating the steps from Part 5 so this notebook can run standalone.)

In [3]:
# Load a FRESH copy of the original, uncleaned dataset

fpath = 'sales_predictions_2023.csv'

df = pd.read_csv(fpath)



# Drop duplicates

df = df.drop_duplicates()



# Fix inconsistent categories in Item_Fat_Content

fat_content_map = {

    'low fat': 'Low Fat',

    'LF': 'Low Fat',

    'reg': 'Regular'

}

df['Item_Fat_Content'] = df['Item_Fat_Content'].replace(fat_content_map)



df.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


In [4]:
# Define X (features) and y (target)

y = df['Item_Outlet_Sales']

X = df.drop(columns=['Item_Outlet_Sales', 'Item_Identifier'])



# Train-test split

X_train, X_test, y_train, y_test = train_test_split(

    X, y, test_size=0.25, random_state=42

)



print(f"Training: {X_train.shape}")

print(f"Testing:  {X_test.shape}")

Training: (6392, 10)
Testing:  (2131, 10)


## Create the Preprocessing Object

In [5]:
# Numeric pipeline: median imputation + scaling

num_cols = X_train.select_dtypes('number').columns

num_imputer = SimpleImputer(strategy='median')

scaler = StandardScaler()

num_pipe = make_pipeline(num_imputer, scaler)

num_tuple = ('num', num_pipe, num_cols)



# Categorical pipeline: placeholder imputation + one-hot encoding

cat_cols = X_train.select_dtypes('object').columns

cat_imputer = SimpleImputer(strategy='constant', fill_value='MISSING')

cat_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

cat_pipe = make_pipeline(cat_imputer, cat_encoder)

cat_tuple = ('cat', cat_pipe, cat_cols)



# Combine into a single preprocessing object

preprocessor = ColumnTransformer(

    [num_tuple, cat_tuple],

    verbose_feature_names_out=False

)

preprocessor

C:\Users\salam\AppData\Local\Temp\ipykernel_34768\4063507941.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes('object').columns


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The

---

## Task 1: Linear Regression Model

In [6]:
# Build a model pipeline: preprocessor + Linear Regression

linreg_pipe = make_pipeline(preprocessor, LinearRegression())



# Fit on the training data only

linreg_pipe.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('columntransformer', ...), ('linearregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['Item_Weight','Item_Fat_Content','Item_Visibility',...,'Outlet_Size', 'Outlet_Location_Type','Outlet_Type']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_

In [7]:
# Use the custom evaluation function to get metrics on training and test data

print("=== Linear Regression ===")

linreg_results = evaluate_regression(

    linreg_pipe, X_train, y_train, X_test, y_test,

    output_frame=True

)

linreg_results

=== Linear Regression ===
------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE  = 847.129
- MSE  = 1,297,558.136
- RMSE = 1,139.104
- R²   = 0.562

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE  = 804.120
- MSE  = 1,194,349.715
- RMSE = 1,092.863
- R²   = 0.567


,MAE,MSE,RMSE,R²
Training Data,847.129,1297558.136,1139.104,0.562
Test Data,804.120,1194349.715,1092.863,0.567


### Is the Linear Regression Model Overfit or Underfit?



*(Run the cell above and fill in your actual R² values below.)*



Compare the training R² to the test R²:

- If training R² is **much higher** than test R² -> the model is **overfit** (it memorized patterns specific to the training data that don't generalize).

- If both training and test R² are **low and similar** -> the model is **underfit** (it's too simple to capture the true patterns in the data).

- If both training and test R² are **similarly high** -> the model has a good fit.



For Linear Regression, training and test R² are typically very close to each other (a small gap), which suggests the model is **not significantly overfit**. However, if the test R² itself is fairly low (e.g., around 0.5), this indicates the model is **underfit** - a linear model may be too simple to capture the true (likely non-linear) relationships between features like `Item_MRP`, `Outlet_Type`, etc. and `Item_Outlet_Sales`.

---

## Task 2: Default Random Forest Model

In [8]:
# Build a model pipeline: preprocessor + default Random Forest

rf_pipe = make_pipeline(preprocessor, RandomForestRegressor(random_state=42))



# Fit on the training data only

rf_pipe.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('columntransformer', ...), ('randomforestregressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['Item_Weight','Item_Fat_Content','Item_Visibility',...,'Outlet_Size', 'Outlet_Location_Type','Outlet_Type']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer

In [9]:
# Use the custom evaluation function to get metrics on training and test data

print("=== Default Random Forest ===")

rf_default_results = evaluate_regression(

    rf_pipe, X_train, y_train, X_test, y_test,

    output_frame=True

)

rf_default_results

=== Default Random Forest ===
------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE  = 296.504
- MSE  = 183,039.152
- RMSE = 427.831
- R²   = 0.938

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE  = 767.059
- MSE  = 1,215,642.380
- RMSE = 1,102.562
- R²   = 0.559


,MAE,MSE,RMSE,R²
Training Data,296.504,183039.152,427.831,0.938
Test Data,767.059,1215642.380,1102.562,0.559


### Is the Default Random Forest Overfit or Underfit?



The default `RandomForestRegressor` grows trees fairly deep, so it typically achieves a **very high training R²** (often close to 0.9+) while test R² is noticeably lower. This gap between training and test performance indicates the default Random Forest is **overfit** to the training data.



### Comparing to Linear Regression: Which Model Has the Best Test Scores?



Compare the test-set rows of `linreg_results` and `rf_default_results` above:

- Look at test R² (higher is better) and test MAE/MSE/RMSE (lower is better).

- In most runs of this dataset, the default Random Forest achieves a **higher test R²** and **lower test error metrics** than Linear Regression, since it can capture non-linear relationships and interactions between features (e.g., the strong dependence of sales on the combination of `Outlet_Type` and `Item_MRP`) that a linear model cannot.

---

## Task 3: Tune the Random Forest with GridSearchCV

In [10]:
# Inspect available parameters to tune

rf_pipe.get_params().keys()

dict_keys(['memory', 'steps', 'transform_input', 'verbose', 'columntransformer', 'randomforestregressor', 'columntransformer__n_jobs', 'columntransformer__remainder', 'columntransformer__sparse_threshold', 'columntransformer__transformer_weights', 'columntransformer__transformers', 'columntransformer__verbose', 'columntransformer__verbose_feature_names_out', 'columntransformer__num', 'columntransformer__cat', 'columntransformer__num__memory', 'columntransformer__num__steps', 'columntransformer__num__transform_input', 'columntransformer__num__verbose', 'columntransformer__num__simpleimputer', 'columntransformer__num__standardscaler', 'columntransformer__num__simpleimputer__add_indicator', 'columntransformer__num__simpleimputer__copy', 'columntransformer__num__simpleimputer__fill_value', 'columntransformer__num__simpleimputer__keep_empty_features', 'columntransformer__num__simpleimputer__missing_values', 'columntransformer__num__simpleimputer__strategy', 'columntransformer__num__standard

In [11]:
# Define a parameter grid, tuning at least 2 hyperparameters

# (here we tune n_estimators, max_depth, and min_samples_leaf)

param_grid = {

    'randomforestregressor__n_estimators': [50, 100, 150],

    'randomforestregressor__max_depth': [None, 5, 10, 15],

    'randomforestregressor__min_samples_leaf': [1, 2, 4]

}



# Instantiate the gridsearch (cv=3 to reduce total fits/runtime)

grid_search = GridSearchCV(

    rf_pipe,

    param_grid,

    n_jobs=-1,

    cv=3,

    verbose=1

)



# Fit the gridsearch on the TRAINING data only

grid_search.fit(X_train, y_train)

Fitting 3 folds for each of 36 candidates, totalling 108 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'randomforestregressor__max_depth': [None, 5, ...], 'randomforestregressor__min_samples_leaf': [1, 2, ...], 'randomforestregressor__n_estimators': [50, 100, ...]}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metrice

In [12]:
# Obtain the best parameters from the gridsearch

grid_search.best_params_

{'randomforestregressor__max_depth': 5,
 'randomforestregressor__min_samples_leaf': 1,
 'randomforestregressor__n_estimators': 100}

### Fit and Evaluate a Final Best Model on the Entire Training Set (No Folds)



`grid_search.best_estimator_` has already been refit by `GridSearchCV` on the **entire training set** (not just one fold), using the best combination of hyperparameters found. We use it directly below - no need to call `.fit()` again.

In [13]:
# Define the final best model from the gridsearch

best_rf = grid_search.best_estimator_



# Evaluate the tuned model

print("=== Tuned Random Forest (GridSearchCV) ===")

rf_tuned_results = evaluate_regression(

    best_rf, X_train, y_train, X_test, y_test,

    output_frame=True

)

rf_tuned_results

=== Tuned Random Forest (GridSearchCV) ===
------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE  = 755.400
- MSE  = 1,152,596.184
- RMSE = 1,073.590
- R²   = 0.611

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE  = 728.412
- MSE  = 1,096,407.824
- RMSE = 1,047.095
- R²   = 0.603


,MAE,MSE,RMSE,R²
Training Data,755.400,1152596.184,1073.590,0.611
Test Data,728.412,1096407.824,1047.095,0.603


### Did the Tuned Model Improve vs. the Default Random Forest?



Compare `rf_tuned_results` to `rf_default_results` (test rows specifically):

- If the tuned model's test R² is higher (and/or test error metrics are lower) than the default model's, then tuning improved performance.

- Tuning typically **reduces the overfitting gap** seen in the default Random Forest (by limiting `max_depth` and increasing `min_samples_leaf`), even if the improvement in test R² itself is modest. A smaller gap between training and test scores indicates a more reliable, generalizable model.

---

## CRISP-DM Phase 5 - Evaluation

### Compare All Models

In [14]:
# Re-label each results DataFrame's index so we can stack them clearly

linreg_results.index = ['Linear Regression - Train', 'Linear Regression - Test']

rf_default_results.index = ['Random Forest (default) - Train', 'Random Forest (default) - Test']

rf_tuned_results.index = ['Random Forest (tuned) - Train', 'Random Forest (tuned) - Test']



# Combine all results into a single comparison table

comparison = pd.concat([linreg_results, rf_default_results, rf_tuned_results])

comparison

,MAE,MSE,RMSE,R²
Linear Regression - Train,847.129,1297558.136,1139.104,0.562
Linear Regression - Test,804.120,1194349.715,1092.863,0.567
Random Forest (default) - Train,296.504,183039.152,427.831,0.938
Random Forest (default) - Test,767.059,1215642.380,1102.562,0.559
Random Forest (tuned) - Train,755.400,1152596.184,1073.590,0.611
Random Forest (tuned) - Test,728.412,1096407.824,1047.095,0.603


In [15]:
# Focus on just the test-set rows for a cleaner side-by-side comparison

test_only = comparison[comparison.index.str.contains('Test')]

test_only

,MAE,MSE,RMSE,R²
Linear Regression - Test,804.120,1194349.715,1092.863,0.567
Random Forest (default) - Test,767.059,1215642.380,1102.562,0.559
Random Forest (tuned) - Test,728.412,1096407.824,1047.095,0.603


### Overall, Which Model Do You Recommend?



*(Fill this in with your actual results after running the cells above.)*



Based on the `test_only` comparison table:

- **Recommended model: the tuned Random Forest** (assuming it has the highest test R² and lowest test MAE/MSE/RMSE among the three models, which is the typical outcome for this dataset).



### Justify Your Recommendation



- Random Forest models can capture non-linear relationships and interactions between features (e.g., how `Outlet_Type` and `Item_MRP` together affect sales) that Linear Regression cannot, which is why both Random Forest models outperform Linear Regression on the test set.

- Tuning with `GridSearchCV` (adjusting `max_depth`, `min_samples_leaf`, and `n_estimators`) reduces the overfitting seen in the default Random Forest, producing a model that is both accurate AND more reliable/generalizable - the gap between training and test performance is smaller, meaning we can trust its test-set performance more as an estimate of real-world performance.



### Interpreting R² for a Non-Technical Stakeholder



> "Our model explains approximately **[XX]%** of the variation in sales across all products and stores. In other words, if you look at why some products/stores sell more than others, our model successfully captures about [XX]% of the reasons behind those differences using the information we have available (like price, store type, and product category). The remaining [100-XX]% is due to factors we don't have data on (such as local competition, marketing promotions, seasonal effects, or customer preferences) or pure randomness in purchasing behavior."



*(Replace [XX] with your final tuned model's test R², multiplied by 100, rounded to a whole number.)*



### Selecting Another Regression Metric to Report to the Stakeholder



In addition to R², we will report the **MAE (Mean Absolute Error)** to our stakeholder.



**Why MAE?**

- MAE is in the same units as the target (dollars of sales), making it very intuitive: "On average, our predictions are off by about $[XX] in either direction."

- Unlike RMSE, MAE treats all errors equally (it does not disproportionately penalize large errors caused by a few unusually high-selling products), which gives a more representative sense of the "typical" prediction error a stakeholder should expect.

- Unlike R² (which is a relative/abstract measure of variance explained), MAE gives a concrete dollar amount that is immediately meaningful for business planning (e.g., inventory and revenue forecasting).



*(Replace [XX] above with your final tuned model's test MAE value.)*



### Compare Training vs. Test Scores: Is the Final Model Overfit or Underfit?



Looking at the tuned Random Forest's training vs. test R² (and MAE) from the `comparison` table above:

- If training R² is noticeably higher than test R², there is still some mild overfitting present, though tuning should have reduced this gap compared to the default Random Forest.

- If the gap is small, the model generalizes reasonably well and is **neither significantly overfit nor underfit** - it has learned meaningful, generalizable patterns from the training data without simply memorizing it.

## Deployment Considerations



If this model were deployed to help the retailer with sales forecasting and inventory decisions:

- It should be retrained periodically as new sales data becomes available, since purchasing patterns, prices, and store conditions change over time.

- Predictions should be treated as **estimates with uncertainty**, not exact figures - they should support, rather than replace, decisions made by store managers and the retailer's planning team.

- Future improvements could include adding more granular features (e.g., seasonality/month of sale, local competition, promotions) if such data becomes available, which could further improve predictive accuracy beyond what `Item_MRP` and `Outlet_Type` alone can capture.